# V2 document preparation and extraction smoke pipeline

This notebook contains the complete V2 preparation workflow: inspect one local or GCS document, run the resumable GCS-to-GCS paragraph pipeline, and optionally execute the production extraction prompts against classified real decisions in a GPU Colab runtime. The paragraph paths convert raw RTF into normalized, globally numbered paragraphs and write:

- `paragraphs.parquet` — one row per paragraph;
- `numbered_document.json` — the complete numbered text and paragraph count.

The paragraph preparation sections require no language model or GPU. The final real-data extraction smoke section is independent, explicitly gated, requires a GPU, and never writes to production GCS or BigQuery resources.

## Dependencies

Use the project environment when running locally. In a clean Colab runtime, uncomment and run the installation command.

In [ ]:
# Colab only:
# %pip install -q "pyarrow>=24,<25" "striprtf>=0.0.32,<0.0.33" "transformers>=5.8,<6" "accelerate>=1.13,<2" "huggingface-hub>=0.34" "json-repair>=0.50,<1" google-cloud-bigquery google-cloud-storage

## Import the V2 paragraph pipeline

The bootstrap supports starting Jupyter from the repository root or any directory beneath it.

In [ ]:
from pathlib import Path
import sys

import pyarrow.parquet as pq
from IPython.display import display

cwd = Path.cwd().resolve()
repository_root = None
source_root = None

for candidate in (cwd, *cwd.parents):
    candidate_source = candidate / "src"
    if (candidate_source / "document_split" / "__init__.py").exists():
        repository_root = candidate
        source_root = candidate_source
        break
    if (candidate / "document_split" / "__init__.py").exists():
        source_root = candidate
        repository_root = candidate.parent
        break

if source_root is None or repository_root is None:
    raise RuntimeError(
        "Could not locate src/document_split. Run the notebook from the "
        "cloned repository or add its src directory to sys.path."
    )

if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from document_split.v2 import (
    DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS,
    DocumentTextParsingSettings,
    V2_INFO_VERSION,
    create_google_cloud_clients,
    parse_document_to_artifacts,
    run_document_text_parsing_pipeline,
)

print(f"Repository: {repository_root}")
print(f"V2 version: {V2_INFO_VERSION}")

## Configure the source document

Use `SOURCE_MODE = "local"` for a file already on disk. Use `SOURCE_MODE = "gcs"` to download the standard `{justice_kind}/{document_id}.rtf` object from Cloud Storage.

In [ ]:
SOURCE_MODE = "local"  # "local" or "gcs"

JUSTICE_KIND = 2
DOCUMENT_ID = "117888886"

# Local source configuration
LOCAL_RTF_PATH = repository_root / f"{DOCUMENT_ID}.rtf"

# GCS authentication. In Colab, store the service-account JSON in a secret
# named cloud_access and grant this notebook access to that secret.
GCS_AUTH_MODE = "colab_secret"  # "colab_secret", "colab_user", or "adc"
COLAB_SERVICE_ACCOUNT_SECRET = "cloud_access"
GCP_PROJECT_ID = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.project_id
BIGQUERY_TABLE = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.bigquery_table

# GCS source configuration; required only when SOURCE_MODE == "gcs"
GCS_SOURCE_BUCKET = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.source_bucket
GCS_SOURCE_OBJECT = f"{JUSTICE_KIND}/{DOCUMENT_ID}.rtf"

# Batch pipeline configuration. Execution remains disabled until explicitly enabled.
RUN_GCS_PIPELINE = False
GCS_SOURCE_PREFIX = ""
GCS_DESTINATION_BUCKET = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.destination_bucket
GCS_DESTINATION_PREFIX = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.destination_prefix
PIPELINE_JUSTICE_KINDS = (2,)
PIPELINE_BATCH_SIZE = 500
PIPELINE_MAX_WORKERS = 5
# Smoke run: process at most 10 unparsed BigQuery rows.
# Set to None only when you are ready for the full run.
PIPELINE_LIMIT = 10
OVERWRITE_EXISTING = False
SHOW_PROGRESS = True

# Local artifacts preserve the V2 cloud-style directory structure.
OUTPUT_DIR = (
    repository_root
    / "artifacts"
    / GCS_DESTINATION_PREFIX
    / str(JUSTICE_KIND)
    / DOCUMENT_ID
)

print(f"Source mode: {SOURCE_MODE}")
print(f"Output directory: {OUTPUT_DIR}")

## Configure Google Cloud authentication

The same credentials are used for BigQuery document selection, source downloads, destination uploads, and `is_parsed` updates.

- `colab_secret` — recommended for this project; reads service-account JSON from the Colab secret configured by `COLAB_SERVICE_ACCOUNT_SECRET`.
- `colab_user` — opens the interactive Colab Google sign-in flow.
- `adc` — uses Application Default Credentials, suitable for a configured local workstation or service account environment.

The metadata-service `404` error occurs when `adc` is used in a runtime that has no attached service account.

In [ ]:
def notebook_google_cloud_clients():
    return create_google_cloud_clients(
        project_id=GCP_PROJECT_ID,
        auth_mode=GCS_AUTH_MODE,
        colab_service_account_secret=COLAB_SERVICE_ACCOUNT_SECRET,
    )

## Load the raw RTF

In [ ]:
if SOURCE_MODE == "local":
    if not LOCAL_RTF_PATH.is_file():
        raise FileNotFoundError(f"RTF file does not exist: {LOCAL_RTF_PATH}")
    raw_rtf = LOCAL_RTF_PATH.read_bytes()
elif SOURCE_MODE == "gcs":
    if not GCS_SOURCE_BUCKET:
        raise ValueError("Set GCS_SOURCE_BUCKET before using GCS mode")
    storage_client = notebook_google_cloud_clients().storage
    raw_rtf = (
        storage_client.bucket(GCS_SOURCE_BUCKET)
        .blob(GCS_SOURCE_OBJECT)
        .download_as_bytes()
    )
else:
    raise ValueError("SOURCE_MODE must be either 'local' or 'gcs'")

print(f"Loaded {len(raw_rtf):,} bytes for document {DOCUMENT_ID}")

## Run the shared RTF-to-artifacts processor

This API performs the same RTF cleanup, paragraph splitting, and global numbering used by both the automated paragraph pipeline and the complete V2 pipeline. It does not use a tokenizer or model.

In [ ]:
state = parse_document_to_artifacts(
    document_id=DOCUMENT_ID,
    justice_kind=JUSTICE_KIND,
    raw_rtf=raw_rtf,
    output_dir=OUTPUT_DIR,
)

paragraphs_path = state.artifact_path("paragraphs.parquet")
numbered_document_path = state.artifact_path("numbered_document.json")

print(f"Paragraphs: {len(state.paragraphs):,}")
print(f"Parquet: {paragraphs_path}")
print(f"Numbered JSON: {numbered_document_path}")

## Validate and inspect the Parquet output

The file intentionally has one row per paragraph. The final extraction pipeline later merges handler results into one document-level row.

In [ ]:
paragraph_table = pq.read_table(paragraphs_path, use_threads=False)

expected_columns = [
    "document_id",
    "paragraph_index",
    "paragraph_order",
    "numbered_text",
    "text",
]
assert paragraph_table.column_names == expected_columns
assert paragraph_table.num_rows == len(state.paragraphs)
assert paragraph_table.column("paragraph_index").to_pylist() == list(
    range(1, paragraph_table.num_rows + 1)
)

print(paragraph_table.schema)
display(paragraph_table.to_pandas().head(20))

## Preview the numbered document

In [ ]:
preview_characters = 5_000
print(state.numbered_text[:preview_characters])
if len(state.numbered_text) > preview_characters:
    print("\n... preview truncated ...")

## Run the automated BigQuery-driven pipeline

The notebook is configured for a 10-document smoke run with `PIPELINE_LIMIT = 10`. Set `RUN_GCS_PIPELINE = True` after configuring authentication, the `document_data` table, and buckets. BigQuery is the source of truth: the pipeline selects at most 10 rows where `is_parsed = FALSE`, builds `{source_prefix}/{justice_kind}/{doc_id}.rtf`, downloads the source RTF, and uses the same batch processing and BigQuery updates as production. It does not scan existing destination objects before starting. After each batch, only successfully uploaded or atomically pre-existing documents are updated to `is_parsed = TRUE`; failures remain unparsed.

```text
document_text_parsing/info_version_9/{justice_kind}/{document_id}/
├── _document.json
├── paragraphs.parquet
└── numbered_document.json
```

A manifest and run logs are stored under `document_text_parsing/info_version_9/`. The Parquet object is uploaded last and acts as the completion marker. After validating the smoke result, set `PIPELINE_LIMIT = None` and rerun this cell for the full dataset.

In [ ]:
if RUN_GCS_PIPELINE:
    pipeline_settings = DocumentTextParsingSettings(
        project_id=GCP_PROJECT_ID,
        bigquery_table=BIGQUERY_TABLE,
        source_bucket=GCS_SOURCE_BUCKET,
        destination_bucket=GCS_DESTINATION_BUCKET,
        source_prefix=GCS_SOURCE_PREFIX,
        destination_prefix=GCS_DESTINATION_PREFIX,
        justice_kinds=PIPELINE_JUSTICE_KINDS,
        batch_size=PIPELINE_BATCH_SIZE,
        max_workers=PIPELINE_MAX_WORKERS,
        limit=PIPELINE_LIMIT,
        overwrite_existing=OVERWRITE_EXISTING,
        show_progress=SHOW_PROGRESS,
    )
    run_label = (
        f"smoke run (limit={PIPELINE_LIMIT})"
        if PIPELINE_LIMIT is not None
        else "full run"
    )
    print(f"Starting {run_label}")
    pipeline_clients = notebook_google_cloud_clients()
    pipeline_result = run_document_text_parsing_pipeline(
        pipeline_settings,
        storage_client=pipeline_clients.storage,
        bigquery_client=pipeline_clients.bigquery,
    )
    print(pipeline_result)
else:
    print(
        "Batch execution disabled. Set RUN_GCS_PIPELINE=True after "
        "configuring authentication and bucket settings."
    )

# Optional GPU stage: real-data extraction smoke test

Run this final stage in a GPU Colab runtime after paragraph classification artifacts are available. It invokes the production V2 prompts locally, without GCS or BigQuery writes. Each document receives an isolated directory containing raw handler responses, normalized JSON, final Parquet, warnings, and a recursive schema-population report.

This stage is independent from the single-RTF cells above; after running the dependency and bootstrap cells, you can jump directly here.

## Select classified real decisions

The expected layout is `<classification root>/<document id>/classification.parquet`. The default three decisions cover probation, imprisonment, civil claims, costs, physical evidence, confiscation, and security measures.

In [ ]:
from document_split.v2 import (
    DEFAULT_V2_EXTRACTION_SETTINGS,
    DEFAULT_V2_PART_PROCESSING_PROMPTS,
    build_sample_processing_contexts,
    run_real_data_smoke,
)

CLASSIFICATION_ROOT = (
    source_root / 'document_split' / 'v2' / 'document_text_parsing' / 'downloads'
)
SMOKE_DOCUMENT_IDS = [
    '118355359',  # probation, costs, evidence
    '118584307',  # imprisonment, civil claim
    '116798672',  # fine, confiscation, security measures
]
CLASSIFICATION_FILES = [
    CLASSIFICATION_ROOT / document_id / 'classification.parquet'
    for document_id in SMOKE_DOCUMENT_IDS
]
SMOKE_OUTPUT_ROOT = Path('/content/v2-real-data-smoke')
RUN_EXTRACTION_SMOKE = False

missing_classifications = [
    path for path in CLASSIFICATION_FILES if not path.is_file()
]
if missing_classifications:
    raise FileNotFoundError(
        'Upload the classified inputs or change CLASSIFICATION_ROOT: '
        + ', '.join(map(str, missing_classifications))
    )

for classification_path in CLASSIFICATION_FILES:
    classification_table = pq.read_table(classification_path)
    part_column = (
        'section'
        if 'section' in classification_table.column_names
        else 'part'
    )
    sections = sorted(set(classification_table[part_column].to_pylist()))
    print(
        classification_path.parent.name,
        'paragraphs=', classification_table.num_rows,
        'sections=', sections,
    )

## Load the extraction model

Select **Runtime → Change runtime type → GPU**. Optionally expose a Colab secret named `HF_TOKEN`. The resolved model commit is printed so the smoke run is reproducible.

In [ ]:
import torch
from huggingface_hub import login, model_info

from document_split.config import MODEL_ID
from document_split.runtime import load_extraction_model

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled Colab runtime is required')

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_REVISION = model_info(MODEL_ID, token=HF_TOKEN).sha
print('Model:', MODEL_ID)
print('Revision:', MODEL_REVISION)
model_pipe, tokenizer = load_extraction_model(
    DEFAULT_V2_EXTRACTION_SETTINGS,
    MODEL_REVISION,
    HF_TOKEN,
)

## Preview routing, batches, and token counts

This prepares the exact production-prompt messages without inference. Review the planned calls before enabling the model run.

In [ ]:
import pandas as pd
from document_split.processing import count_message_tokens

smoke_preview_rows = []
for classification_path in CLASSIFICATION_FILES:
    contexts = build_sample_processing_contexts(
        document_id=classification_path.parent.name,
        justice_kind=2,
        parts_parquet_bytes=classification_path.read_bytes(),
        tokenizer=tokenizer,
        part_prompts=DEFAULT_V2_PART_PROCESSING_PROMPTS,
    )
    for processor_name, batches in contexts.items():
        for batch_number, context in enumerate(batches, start=1):
            smoke_preview_rows.append({
                'document_id': classification_path.parent.name,
                'processor': processor_name,
                'batch': batch_number,
                'target_paragraphs': len(context.target_paragraph_ids),
                'input_tokens': count_message_tokens(
                    tokenizer, context.messages
                ),
            })

smoke_preview_df = pd.DataFrame(smoke_preview_rows)
display(smoke_preview_df)
print('Planned model calls:', len(smoke_preview_df))

## Run isolated inference

After inspecting the preview, set `RUN_EXTRACTION_SMOKE = True` in the configuration cell and run this cell.

In [ ]:
if not RUN_EXTRACTION_SMOKE:
    raise RuntimeError(
        'Review the preview, then set RUN_EXTRACTION_SMOKE = True'
    )

smoke_results = run_real_data_smoke(
    classification_files=CLASSIFICATION_FILES,
    output_root=SMOKE_OUTPUT_ROOT,
    model_pipe=model_pipe,
    tokenizer=tokenizer,
)
print('Smoke artifacts:', SMOKE_OUTPUT_ROOT)

## Review population and expected anchors

Population indicates which schema paths received non-empty values. The source-specific checks catch obvious omissions, but passing them does not replace manual comparison with the decision text.

In [ ]:
import json

population_rows = []
smoke_payloads = {}
for smoke_result in smoke_results:
    for schema_path, statistics in smoke_result.population.items():
        population_rows.append({
            'document_id': smoke_result.document_id,
            'path': schema_path,
            **statistics,
        })
    smoke_payloads[smoke_result.document_id] = json.loads(
        (smoke_result.artifact_dir / 'result.json').read_text(
            encoding='utf-8'
        )
    )

population_df = pd.DataFrame(population_rows)
display(
    population_df[population_df['populated']].sort_values(
        ['document_id', 'path']
    )
)

def nested_value(value, dotted_path):
    for key in dotted_path.split('.'):
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value

def contains_article(value, article):
    if isinstance(value, dict):
        if str(value.get('article')) == article:
            return True
        return any(
            contains_article(child, article) for child in value.values()
        )
    if isinstance(value, list):
        return any(contains_article(child, article) for child in value)
    return False

EXPECTED_ANCHORS = {
    '118355359': {
        'article': '286',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.probation',
            'operative_part.costs_reimbursement_decision',
            'operative_part.physical_evidence_decision',
        ],
    },
    '118584307': {
        'article': '185',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.sentence_start',
            'operative_part.civil_claim_decision',
        ],
    },
    '116798672': {
        'article': '369-2',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.security_measures_decision',
            'operative_part.physical_evidence_decision',
        ],
    },
}

anchor_rows = []
for smoke_document_id, expected in EXPECTED_ANCHORS.items():
    if smoke_document_id not in smoke_payloads:
        continue
    payload = smoke_payloads[smoke_document_id]
    anchor_rows.append({
        'document_id': smoke_document_id,
        'expectation': f"article {expected['article']}",
        'passed': contains_article(payload, expected['article']),
    })
    for expected_path in expected['paths']:
        extracted_value = nested_value(payload, expected_path)
        anchor_rows.append({
            'document_id': smoke_document_id,
            'expectation': expected_path,
            'passed': extracted_value not in (None, '', []),
        })

anchor_df = pd.DataFrame(anchor_rows)
display(anchor_df)
print('Anchor checks passed:', bool(anchor_df['passed'].all()))

for smoke_document_id, payload in smoke_payloads.items():
    print('\n===', smoke_document_id, '===')
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])

## Download smoke artifacts

Download the ZIP before the Colab runtime is recycled.

In [ ]:
import shutil

smoke_archive = shutil.make_archive(
    '/content/v2-real-data-smoke',
    'zip',
    SMOKE_OUTPUT_ROOT,
)
try:
    from google.colab import files
    files.download(smoke_archive)
except ImportError:
    print('Archive:', smoke_archive)